[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch09-data-selection/02-cat-tia-tinh.ipynb)

# Cắt tỉa tĩnh: chấm điểm từng mẫu, chọn tập lõi, loại bỏ trùng lặp

Chặng đầu của chọn lọc dữ liệu chạy trọn vẹn **trước** khi huấn luyện. Nó không đổi vòng lặp hay
kiến trúc, nhưng chương nhắc nhiều lần rằng tác động lên chất lượng cuối phải được đo. Sổ tay này đo
đúng điều đó trên FashionMNIST, với một mô hình thay thế (proxy model) nhỏ làm bộ chấm điểm.

**Bạn sẽ làm:**
1. huấn luyện một mô hình thay thế 5 epoch và chấm mọi mẫu bằng ba điểm số: EL2N, mất mát và số lần
   quên (forgetting events);
2. giữ 10 tới 70 phần trăm tập theo điểm số hoặc ngẫu nhiên, huấn luyện lại, kiểm tra độ phủ theo lớp
   và tính DCR;
3. chọn tập lõi (coreset) bằng k-Center trên một biểu diễn PCA, so với chọn ngẫu nhiên;
4. kiểm bất đẳng thức chọn lọc bằng thời gian đo trên máy, rồi tính lại ví dụ 1.5 và hai cạm bẫy của
   chương: phụ trội chọn lọc và lớp hiếm;
5. loại bỏ trùng lặp (deduplication) ở ba mức: băm mật mã cho bản trùng từng byte, băm cảm nhận cho ảnh
   gần trùng, MinHash cho văn bản gần trùng.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import hashlib
import zlib

import torch.nn.functional as F
import torchvision
from scipy.stats import spearmanr

tap_hl = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
tap_kt = torchvision.datasets.FashionMNIST(DATA, train=False, download=True)
N, N_KT = 10_000, 5_000
anh = tap_hl.data[:N].clone()  # uint8, giữ lại cho phần loại bỏ trùng lặp
X = anh.reshape(-1, 784).float() / 255
y = tap_hl.targets[:N].clone()
X_kt = tap_kt.data[:N_KT].reshape(-1, 784).float() / 255
y_kt = tap_kt.targets[:N_KT].clone()
TEN_LOP = ["áo thun", "quần dài", "áo len", "váy", "áo khoác", "xăng-đan", "áo sơ mi", "giày thể thao", "túi", "bốt"]
assert len(tap_hl.classes) == len(TEN_LOP)
print(f"{N:,} ảnh huấn luyện, {N_KT:,} ảnh kiểm tra; mỗi lớp có {torch.bincount(y).min()}–{torch.bincount(y).max()} ảnh")


def mlp(an=128):
    return torch.nn.Sequential(torch.nn.Linear(784, an), torch.nn.ReLU(), torch.nn.Linear(an, 10))


def huan_luyen(chon, so_epoch=15, batch=128, an=128, seed=SEED, moi_epoch=None):
    """Huấn luyện một MLP trên các mẫu có chỉ số `chon`. moi_epoch(m) chạy sau mỗi epoch."""
    dat_hat_giong(seed)
    m = mlp(an)
    toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
    for _ in range(so_epoch):
        thu_tu = chon[torch.randperm(len(chon))]
        for i in range(0, len(thu_tu), batch):
            b = thu_tu[i:i + batch]
            mat_mat = F.cross_entropy(m(X[b]), y[b])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
        if moi_epoch:
            moi_epoch(m)
    return m


def do_chinh_xac(m):
    with torch.no_grad():
        return 100 * (m(X_kt).argmax(1) == y_kt).float().mean().item()


def mac_xuoi(an):
    return 784 * an + an * 10

## 1. Ba điểm số từ một mô hình thay thế

Ví dụ 1.1 của chương chọn 100,000 ảnh, tức 10 phần trăm của 1 triệu ảnh, sau khi huấn luyện mô hình
thay thế 5 epoch để tính điểm. Sổ tay làm cùng việc ở quy mô nhỏ: một MLP 784-32-10, nhỏ hơn mô hình
đích 784-128-10 bốn lần, học 5 epoch trên cả 10,000 ảnh. Từ nó ta lấy ba điểm số:

* **EL2N**: khoảng cách L2 giữa xác suất dự đoán và nhãn one-hot. Điểm cao là mẫu khó với mô hình thay
  thế.
* **mất mát** entropy chéo (cross-entropy) của mẫu, một bộ chấm độ khó theo mất mát.
* **số lần quên**: số lần một mẫu đang được phân loại đúng rồi bị phân loại sai. Sổ tay chỉ kiểm sau mỗi
  epoch, nên cách đếm thô hơn kiểm sau mỗi lần cập nhật.

In [ ]:
EPOCH_PROXY = sach(5, nguon="100,000 · 10 phần trăm · 1 triệu · 5 epoch")
theo_sach("tập lõi của ví dụ 1.1", sach(10, nguon="100,000 · 10 phần trăm") / 100 * sach(1e6, nguon="10 phần trăm · 1 triệu"),
          sach=100_000, nguon="100,000 · 10 phần trăm · 1 triệu · 5 epoch")

dung_moi_epoch = []


def ghi_dung_sai(m):
    with torch.no_grad():
        dung_moi_epoch.append(m(X).argmax(1) == y)


TAT_CA = torch.arange(N)
t0 = time.perf_counter()
proxy = huan_luyen(TAT_CA, so_epoch=EPOCH_PROXY, an=32, moi_epoch=ghi_dung_sai)
with torch.no_grad():
    logit = proxy(X)
el2n = (logit.softmax(1) - F.one_hot(y, 10)).norm(dim=1)
mat_mat_mau = F.cross_entropy(logit, y, reduction="none")
bang_dung = torch.stack(dung_moi_epoch)  # epoch × mẫu
so_lan_quen = (bang_dung[:-1] & ~bang_dung[1:]).sum(0)
t_cham_nhe = time.perf_counter() - t0
do_tren_may("thời gian huấn luyện mô hình thay thế và chấm 10,000 mẫu", t_cham_nhe, "s")

print(f"mẫu bị quên ít nhất một lần: {int((so_lan_quen > 0).sum()):,}")
print(f"mẫu chưa từng được phân loại đúng: {int((~bang_dung.any(0)).sum()):,}")
r1, r2 = spearmanr(el2n, mat_mat_mau).statistic, spearmanr(el2n, so_lan_quen).statistic
do_tren_may("tương quan hạng EL2N và mất mát", r1)
do_tren_may("tương quan hạng EL2N và số lần quên", r2)

In [ ]:
thu_tu = torch.argsort(el2n)  # dễ → khó
fig, axes = plt.subplots(2, 8, figsize=(12, 3.9))
for hang, (ds, ten) in enumerate(((thu_tu[:8], "EL2N thấp nhất"), (thu_tu.flip(0)[:8], "EL2N cao nhất"))):
    for ax, i in zip(axes[hang], ds.tolist()):
        ax.imshow(X[i].reshape(28, 28), cmap="gray_r")
        ax.set_title(f"{TEN_LOP[y[i]]}\nEL2N {el2n[i]:.2f}", fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    axes[hang][0].set_ylabel(ten, fontsize=9)
fig.suptitle("Nhãn ghi trong tập, và điểm EL2N của mô hình thay thế", fontsize=11)
plt.tight_layout(h_pad=1.5)
plt.show()

EL2N và mất mát gần như xếp mẫu theo cùng một thứ tự, vì cả hai đều đo mô hình còn sai bao xa với nhãn.
Số lần quên đo một thứ khác: mẫu nằm gần biên quyết định thì bị lật qua lật lại giữa các epoch. Hàng
trên toàn là mẫu điển hình, dễ. Hàng dưới là mẫu khó, nhưng hãy nhìn kỹ: vài mẫu trông như mang nhãn
sai. Chương nói đúng điều đó: điểm cao là mẫu khó với mô hình thay thế, nhưng cũng có thể là nhiễu.

## 2. Giữ phần nào của tập?

Giữ một tỉ lệ của tập theo bốn chính sách, rồi huấn luyện lại mô hình đích 784-128-10 với cùng 15 epoch:

* **ngẫu nhiên**, mốc so sánh;
* **khó nhất**: EL2N cao nhất;
* **dễ nhất**: EL2N thấp nhất;
* **khó nhất, chia đều theo lớp**: cùng điểm số, nhưng mỗi lớp giữ đúng một phần mười ngân sách. Đây là
  rào chắn chương đặt ra trước khi áp bất kỳ điểm số nào: mỗi lớp phải có mức đại diện tối thiểu.

Lịch cố định nên phần dữ liệu giữ lại cũng chính là phần FLOP huấn luyện phải trả.

**Dự đoán:** ở mức giữ 10 phần trăm, chính sách nào cho độ chính xác cao nhất? Ở 70 phần trăm thì sao?

In [ ]:
TI_LE = [0.1, 0.3, 0.5, 0.7]
tron = torch.randperm(N, generator=torch.Generator().manual_seed(SEED))


def kho_nhat_theo_lop(k):
    return torch.cat([thu_tu[y[thu_tu] == c][-(k // 10):] for c in range(10)])


CHINH_SACH = {
    "ngẫu nhiên": lambda k: tron[:k],
    "khó nhất": lambda k: thu_tu[-k:],
    "dễ nhất": lambda k: thu_tu[:k],
    "khó nhất, chia đều theo lớp": kho_nhat_theo_lop,
}
t0 = time.perf_counter()
dung_ca_tap = do_chinh_xac(huan_luyen(TAT_CA))
t_ca_tap = time.perf_counter() - t0
do_tren_may("cả 10,000 ảnh: độ chính xác", dung_ca_tap, "%")

ket_qua, tap_con, thoi_gian_con = {}, {}, {}
for ten, chon_k in CHINH_SACH.items():
    ket_qua[ten] = []
    for f in TI_LE:
        chon = chon_k(int(f * N))
        assert len(chon) == int(f * N) and len(set(chon.tolist())) == len(chon)
        t0 = time.perf_counter()
        ket_qua[ten].append(do_chinh_xac(huan_luyen(chon)))
        thoi_gian_con[(ten, f)] = time.perf_counter() - t0
        tap_con[(ten, f)] = chon
    print(f"⏱ {ten:<28} giữ 10 / 30 / 50 / 70 phần trăm: "
          + " / ".join(f"{a:.1f}" for a in ket_qua[ten]) + " % (đo trên máy này)")

In [ ]:
MAU = {"ngẫu nhiên": "0.35", "khó nhất": "C3", "dễ nhất": "C0", "khó nhất, chia đều theo lớp": "C2"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
for ten, kq in ket_qua.items():
    ax1.plot([100 * f for f in TI_LE] + [100], kq + [dung_ca_tap], "o-", color=MAU[ten], lw=2, label=ten)
ax1.axhline(dung_ca_tap, color="0.6", lw=0.8, ls=":")
ax1.text(11, dung_ca_tap + 0.8, "cả tập", fontsize=8, color="0.35")
ax1.set_xlabel("phần dữ liệu giữ lại (%) = phần FLOP huấn luyện, lịch cố định")
ax1.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax1.set_title("Cắt tỉa theo EL2N (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)

f_xem = 0.1
rong = 0.2
for j, ten in enumerate(CHINH_SACH):
    dem = torch.bincount(y[tap_con[(ten, f_xem)]], minlength=10).numpy()
    ax2.bar(np.arange(10) + (j - 1.5) * rong, dem, width=rong, color=MAU[ten], label=ten)
ax2.axhline(f_xem * N / 10, color="0.3", lw=0.8, ls="--")
ax2.text(9.6, f_xem * N / 10 + 6, "chia đều", fontsize=8, ha="right", color="0.3")
ax2.set_xticks(range(10), TEN_LOP, rotation=40, ha="right", fontsize=8)
ax2.set_ylabel("số ảnh trong tập con")
ax2.set_title(f"Độ phủ theo lớp khi giữ {f_xem:.0%} (đo trên máy này)")
ax2.set_ylim(0, 1.5 * max(torch.bincount(y[tap_con[(ten, f_xem)]], minlength=10).max().item() for ten in CHINH_SACH))
ax2.legend(loc="upper center", fontsize=7, ncol=2)  # nửa trên trục để trống cho chú giải
plt.tight_layout()
plt.show()

So các đường ở đầu trái. "Khó nhất" nằm dưới "ngẫu nhiên" rất xa, và biểu đồ cột bên phải cho biết vì sao:
chính sách ấy dồn ngân sách vào vài lớp dễ nhầm với nhau và gần như bỏ trống những lớp dễ. Nó còn gom luôn
những mẫu nhãn sai, vì chúng cũng có EL2N cao. Rào chắn chia đều theo lớp sửa được một phần, nhưng chưa đủ.
Chọn ngẫu nhiên là một mốc mạnh một cách bất ngờ, đúng như chương nói: kết quả của các bài báo gắn với từng
benchmark, nên phải kiểm định trên đúng tác vụ trước khi cắt mạnh tay.

Ở đầu phải thứ hạng đảo lại: khi giữ một nửa tập trở lên, "khó nhất" theo kịp rồi vượt "ngẫu nhiên", vì phần
bị bỏ giờ là những mẫu dễ, dư thừa. Còn "dễ nhất" nằm dưới ở mọi mức giữ: những ảnh điển hình nhất gần như
chỉ dạy lại điều mô hình đã biết. Không chính sách nào thắng ở mọi ngân sách.

### Tỉ số nén dữ liệu

Phần khung đo lường của chương định nghĩa $\text{DCR} = D_{\text{đủ}} / D_{\text{tập lõi}}$, đo ở một độ
chính xác mục tiêu. DCR 5 lần nghĩa là tập lõi đạt mục tiêu với 20 phần trăm dữ liệu. Sổ tay đặt mục tiêu
là độ chính xác của cả tập trừ 2 điểm, một lựa chọn của sổ tay.

In [ ]:
theo_sach("DCR khi tập lõi chỉ cần 20% dữ liệu", 1 / (sach(20, nguon="5× · 20 phần trăm") / 100), sach=5, nguon="5× · 20 phần trăm")
MUC_TIEU = dung_ca_tap - 2
x = np.array(TI_LE + [1.0])
for ten, kq in ket_qua.items():
    duong = np.maximum.accumulate(np.array(kq + [dung_ca_tap]))
    if duong[0] >= MUC_TIEU:
        print(f"{ten:<28} đạt mục tiêu ngay ở mức giữ nhỏ nhất đã thử")
        continue
    f_can = float(np.interp(MUC_TIEU, duong, x)) if duong[-1] >= MUC_TIEU else 1.0
    do_tren_may(f"{ten}: DCR ở mục tiêu {MUC_TIEU:.1f}%", 1 / f_can, "×")

## 3. Tập lõi theo hình học: k-Center

Phương pháp hình học chọn mẫu phủ một biểu diễn đã chọn, mà không cần huấn luyện mô hình đích. k-Center
chọn từng tâm một, mỗi lần lấy điểm đang xa tâm gần nhất của nó nhất, để thu nhỏ khoảng cách lớn nhất từ
mọi điểm tới tâm được chọn gần nhất. Biểu diễn ở đây là 32 thành phần chính (PCA) của pixel, không dùng
nhãn.

Bảng 3 ghi chi phí của k-Center là $\mathcal{O}(D^2)$ hoặc $\mathcal{O}(DK)$. Bản tham lam dưới đây là
$\mathcal{O}(DK)$: mỗi tâm mới chỉ cần cập nhật khoảng cách của $D$ điểm tới nó.

In [ ]:
Xc = X - X.mean(0)
_, _, V = torch.linalg.svd(Xc, full_matrices=False)
Z = Xc @ V[:32].T


def k_center(K, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    chon = [int(torch.randint(N, (1,), generator=g))]
    gan_nhat = ((Z - Z[chon[0]]) ** 2).sum(1)
    for _ in range(K - 1):
        j = int(gan_nhat.argmax())
        chon.append(j)
        gan_nhat = torch.minimum(gan_nhat, ((Z - Z[j]) ** 2).sum(1))
    return torch.tensor(chon)


def ban_kinh(chon):
    d = torch.cdist(Z, Z[chon]).min(1).values
    return d.max().item(), d.mean().item()


CO = [100, 200, 500, 1_000]
kc = {"ngẫu nhiên": [], "k-Center": []}
phu = {"ngẫu nhiên": [], "k-Center": []}
lop_it_nhat = {"ngẫu nhiên": [], "k-Center": []}
for K in CO:
    for ten, chon in (("ngẫu nhiên", tron[:K]), ("k-Center", k_center(K))):
        kc[ten].append(do_chinh_xac(huan_luyen(chon, so_epoch=int(np.ceil(6000 / K)), batch=32)))
        phu[ten].append(ban_kinh(chon))
        lop_it_nhat[ten].append(int(torch.bincount(y[chon], minlength=10).min()))
    print(f"K = {K:>5,}: phép tính khoảng cách {N * K:>12,} (O(DK)) so với {N * N:,} (O(D²))")
for ten in kc:
    print(f"⏱ {ten}: độ chính xác ở K = 100 / 200 / 500 / 1,000: "
          + " / ".join(f"{a:.1f}" for a in kc[ten]) + " % (đo trên máy này)")
    print(f"   lớp ít ảnh nhất trong tập con: {lop_it_nhat[ten]}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for ten, mau in (("ngẫu nhiên", "0.35"), ("k-Center", "C4")):
    ax1.semilogx(CO, kc[ten], "o-", color=mau, lw=2, label=ten)
    ax2.semilogx(CO, [p[0] for p in phu[ten]], "o-", color=mau, lw=2, label=f"{ten}: xa nhất")
    ax2.semilogx(CO, [p[1] for p in phu[ten]], "o--", color=mau, lw=1.5, label=f"{ten}: trung bình")
for ax in (ax1, ax2):
    ax.set_xticks(CO, [f"{k:,}" for k in CO])
    ax.minorticks_off()
    ax.set_xlabel("số ảnh trong tập lõi K (thang log)")
ax1.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax1.set_title("Huấn luyện trên tập lõi (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)
ax2.set_ylabel("khoảng cách tới tâm gần nhất (PCA)")
ax2.set_title("k-Center thu nhỏ khoảng cách xa nhất")
ax2.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

k-Center làm đúng việc nó được thiết kế: khoảng cách **xa nhất** nhỏ hơn hẳn chọn ngẫu nhiên. Nhưng để
làm vậy nó phải lấy những điểm nằm rìa, tức những ảnh lạ nhất, nên khoảng cách **trung bình** có thể lại
lớn hơn, và số ảnh của lớp ít nhất có thể rất thấp vì điểm số không biết tới nhãn. Độ chính xác của mô
hình đích phụ thuộc vào cả hai, nên thứ hạng giữa hai cách chọn tuỳ tập dữ liệu và biểu diễn. Đó là lý do
chương dặn hai họ phương pháp tối ưu những đại lượng khác nhau, và bảng 3 ghi giới hạn của k-Center là
"bỏ qua thông tin nhãn".

## 4. Bất đẳng thức chọn lọc

Chọn lọc chỉ đáng khi
$T_{\text{chọn}} + T_{\text{huấn luyện}}(D_{\text{tập con}}) < T_{\text{huấn luyện}}(D_{\text{toàn bộ}})$.
Phần 2 đã đo đủ các số hạng trên máy của bạn. Đặt thêm một hàm chọn **đắt**: đếm số lần quên bằng cả
một lần huấn luyện mô hình đích đầy đủ, đúng như bảng 3 ghi chi phí của Forgetting.

**Dự đoán:** với tập con 50 phần trăm, cách chọn đắt có còn tiết kiệm được gì không?

In [ ]:
t_con = thoi_gian_con[("khó nhất", 0.5)]
t_chon_nang = t_ca_tap  # một lần huấn luyện đầy đủ để đếm số lần quên
thanh = {"huấn luyện cả tập": [("huấn luyện cả tập", t_ca_tap)],
         "chọn nhẹ (mô hình thay thế 5 epoch)": [("chọn", t_cham_nhe), ("huấn luyện 50%", t_con)],
         "chọn nặng (cả một lần huấn luyện)": [("chọn", t_chon_nang), ("huấn luyện 50%", t_con)]}
for ten, phan in thanh.items():
    do_tren_may(f"{ten}: tổng", sum(t for _, t in phan), "s")
print("bất đẳng thức chọn lọc, chọn nhẹ:", "thoả" if t_cham_nhe + t_con < t_ca_tap else "không thoả")
print("bất đẳng thức chọn lọc, chọn nặng:", "thoả" if t_chon_nang + t_con < t_ca_tap else "không thoả")

# Cùng phép so, tính bằng FLOP thay vì giây (một bước huấn luyện bằng 3 lượt xuôi, như chương).
flop_hl = lambda so_mau, so_epoch, an: 3 * 2 * mac_xuoi(an) * so_mau * so_epoch
flop_ca_tap = flop_hl(N, 15, 128)
flop_chon_nhe = flop_hl(N, EPOCH_PROXY, 32) + 2 * mac_xuoi(32) * N  # huấn luyện + một lượt chấm
print(f"FLOP: chọn nhẹ bằng {flop_chon_nhe / flop_ca_tap:.1%} một lần huấn luyện đầy đủ")
do_tren_may("thời gian: chọn nhẹ so với một lần huấn luyện đầy đủ", 100 * t_cham_nhe / t_ca_tap, "%")

fig, ax = plt.subplots(figsize=(8, 2.8))
for hang, (ten, phan) in enumerate(thanh.items()):
    trai = 0
    for nhan, t in phan:
        mau = "0.55" if nhan == "huấn luyện cả tập" else ("C1" if nhan == "chọn" else "C0")
        ax.barh(hang, t, left=trai, color=mau, edgecolor="white")
        ax.text(trai + t / 2, hang, nhan, ha="center", va="center", fontsize=8, color="white")
        trai += t
ax.axvline(t_ca_tap, color="0.2", lw=1, ls="--")
ax.set_yticks(range(3), list(thanh))
ax.invert_yaxis()
ax.set_xlabel("giây (đo trên máy này)")
ax.set_title("Ba thanh thời gian: chọn lọc phải rẻ hơn phần nó tiết kiệm")
plt.tight_layout()
plt.show()

Thanh thứ ba luôn dài hơn thanh thứ nhất: một hàm chọn cần cả một lần huấn luyện đầy đủ thì không bao
giờ tiết kiệm được cho chính lần huấn luyện ấy. Thanh thứ hai mới là lời hứa của chọn lọc. Để ý thêm một
điều mà phép đếm FLOP không cho thấy: trên CPU, một mô hình nhỏ hơn bốn lần không chạy nhanh hơn bốn lần,
vì mỗi bước còn phụ trội cố định. Chi phí chọn phải được đo, không chỉ đếm.

### Ví dụ 1.5 và cạm bẫy phụ trội chọn lọc

Ví dụ 1.5 chọn tập lõi 10 phần trăm từ 1 triệu ảnh. Phương án A chấm bằng chính ResNet-50, phương án B
bằng ResNet-18 làm mô hình thay thế.

In [ ]:
PHU_TROI_A = sach(2.8, nguon="100 nghìn · 1 triệu · 2.8 giờ · 247.2 giờ")
TIET_KIEM_A = sach(247.2, nguon="100 nghìn · 1 triệu · 2.8 giờ · 247.2 giờ")
PHU_TROI_B = sach(0.6, nguon="0.6 giờ · 249.4 giờ")
TIET_KIEM_B = sach(249.4, nguon="0.6 giờ · 249.4 giờ")
# Hai phương án huấn luyện cùng một tập con, nên B tiết kiệm thêm đúng phần phụ trội nó bớt được.
them = TIET_KIEM_B - TIET_KIEM_A
assert abs(them - (PHU_TROI_A - PHU_TROI_B)) < 1e-9
print(f"B tiết kiệm thêm {them:.1f} giờ = phụ trội A {PHU_TROI_A} giờ − phụ trội B {PHU_TROI_B} giờ")
print(f"phụ trội của A nhiều gấp {PHU_TROI_A / PHU_TROI_B:.1f} lần của B")

# Cạm bẫy: lần chạy đầy đủ 8 giờ, huấn luyện trên tập lõi 2 giờ, thuật toán chọn 10 giờ.
DAY_DU = sach(8, nguon="8 giờ · 2 giờ · 10 giờ · 5× · 30 phút · 6.2 phần trăm")
TAP_LOI = sach(2, nguon="8 giờ · 2 giờ · 10 giờ")
CHON = sach(10, nguon="8 giờ · 2 giờ · 10 giờ")
theo_sach("thuật toán chọn so với lần chạy nó phục vụ", CHON / TAP_LOI, sach=5, nguon="8 giờ · 2 giờ · 10 giờ · 5×")
print(f"chọn + tập lõi = {CHON + TAP_LOI} giờ, so với {DAY_DU} giờ huấn luyện cả tập:",
      "thoả" if CHON + TAP_LOI < DAY_DU else "không thoả bất đẳng thức chọn lọc")
EL2N_PHUT = sach(30, nguon="5× · 30 phút · 6.2 phần trăm")
ti_le = 100 * EL2N_PHUT / 60 / DAY_DU
theo_sach("chấm EL2N bằng mô hình thay thế, so với lần chạy đầy đủ (%)", ti_le, sach=6.2,
          nguon="30 phút · 6.2 phần trăm")
print(f"tính đúng là {ti_le:.2f} phần trăm; sách in 6.2")

30 phút chia 8 giờ là đúng 6.25 phần trăm, sách in 6.2: một cách làm tròn xuống ở chữ số thứ hai, không
đổi kết luận. Cả hai cạm bẫy nói cùng một điều với các thanh thời gian ở trên: thuật toán chọn tinh vi nhất
cũng vô ích nếu nó tốn hơn phần huấn luyện mà nó tiết kiệm.

## 5. Lớp hiếm, và chọn cục bộ trên từng shard

Cạm bẫy thứ ba của chương: trong 1 triệu mẫu có 0.1 phần trăm thuộc lớp hiếm, tức 1,000 ví dụ. Một tập con
10 phần trăm không phân tầng chỉ giữ trung bình 100 ví dụ, dưới mục tiêu 150 của kịch bản.

**Dự đoán:** nếu bốc ngẫu nhiên tập con 10 phần trăm thật nhiều lần, bao nhiêu lần may mắn đạt 150?

In [ ]:
N_LON = sach(1e6, nguon="1 triệu · 0.1 phần trăm · 1,000 · 10 phần trăm · 100 · 150")
TI_LE_HIEM = sach(0.1, nguon="1 triệu · 0.1 phần trăm") / 100
GIU_CON = sach(10, nguon="1,000 · 10 phần trăm · 100 · 150") / 100
MUC_HIEM = sach(150, nguon="1,000 · 10 phần trăm · 100 · 150")
so_hiem = round(N_LON * TI_LE_HIEM)
theo_sach("ví dụ của lớp hiếm", so_hiem, sach=1_000, nguon="1 triệu · 0.1 phần trăm · 1,000")
theo_sach("số giữ lại trung bình", so_hiem * GIU_CON, sach=100, nguon="1,000 · 10 phần trăm · 100 · 150")

rng = np.random.default_rng(SEED)
LAN = 200_000
dem = rng.hypergeometric(so_hiem, int(N_LON) - so_hiem, int(GIU_CON * N_LON), size=LAN)
print(f"{LAN:,} lần bốc: trung bình {dem.mean():.1f}, độ lệch chuẩn {dem.std():.1f}, lớn nhất {dem.max()}")
print(f"số lần đạt {MUC_HIEM}: {(dem >= MUC_HIEM).sum()}")
z = (MUC_HIEM - so_hiem * GIU_CON) / dem.std()
print(f"mục tiêu nằm cách trung bình {z:.1f} độ lệch chuẩn")

fig, ax = plt.subplots(figsize=(7, 3.2))
cot, _, _ = ax.hist(dem, bins=np.arange(dem.min(), max(dem.max(), MUC_HIEM) + 2), color="C0", alpha=0.8)
ax.set_ylim(0, 1.3 * cot.max())  # chừa chỗ phía trên cho nhãn
ax.axvline(MUC_HIEM, color="C3", lw=2)
ax.text(MUC_HIEM - 1, 1.15 * cot.max(), "mục tiêu 150", color="C3", ha="right", fontsize=9)
ax.axvline(so_hiem * GIU_CON, color="0.2", lw=1, ls="--")
ax.text(so_hiem * GIU_CON + 1, 1.15 * cot.max(), "kỳ vọng 100", color="0.2", fontsize=9)
ax.set_xlabel("số ví dụ lớp hiếm trong tập con 10 phần trăm")
ax.set_ylabel("số lần bốc")
ax.set_title("Chọn ngẫu nhiên không phân tầng không bao giờ chạm mục tiêu")
plt.tight_layout()
plt.show()

Không lần nào chạm 150: mục tiêu nằm cách trung bình hơn năm độ lệch chuẩn. Đây không phải chuyện xui. Giải
pháp của chương là đặt **mức đại diện tối thiểu theo lớp trước**, rồi mới tối ưu phần ngân sách còn lại.

### Khi dữ liệu bị chia shard

Huấn luyện phân tán chia dữ liệu cho nhiều worker, và mỗi worker chỉ thấy shard của mình. Dữ liệu thật
thường không chia đều: mỗi nguồn thu thập cho ra một kiểu ảnh. Mô phỏng điều đó bằng 8 shard xếp theo lớp,
rồi so hai cách giữ 30 phần trăm mẫu khó nhất: mỗi worker tự giữ 30 phần trăm shard của mình, hoặc một nút
điều phối xếp hạng toàn cục.

In [ ]:
SO_SHARD = 8
theo_lop = torch.argsort(y * N + torch.arange(N))  # xếp mẫu theo lớp, như 8 nguồn thu thập khác nhau
shard = torch.empty(N, dtype=torch.long)
shard[theo_lop] = torch.arange(N) * SO_SHARD // N
K_GIU = int(0.3 * N)
toan_cuc = set(thu_tu[-K_GIU:].tolist())
cuc_bo = set()
for s_ in range(SO_SHARD):
    trong = (shard == s_).nonzero().squeeze(1)
    k = round(0.3 * len(trong))
    cuc_bo |= set(trong[torch.argsort(el2n[trong])[-k:]].tolist())
chung = len(toan_cuc & cuc_bo) / len(toan_cuc | cuc_bo)
do_tren_may("độ trùng Jaccard giữa chọn cục bộ và chọn toàn cục", chung)

# Ngưỡng toàn cục từ thống kê phân phối điểm: mỗi worker chỉ gửi một histogram điểm EL2N của shard mình.
# EL2N nằm trong [0, √2], nên mọi worker dùng chung một lưới ô mà không cần hỏi ai.
SO_O = 256
bien = torch.linspace(0, math.sqrt(2) + 1e-4, SO_O + 1)
tong = torch.zeros(SO_O)
for s_ in range(SO_SHARD):
    tong += torch.histogram(el2n[shard == s_], bins=bien).hist
tu_tren = tong.flip(0).cumsum(0)  # số mẫu từ ô cao nhất trở xuống
o = int((tu_tren >= K_GIU).nonzero()[0])
nguong = bien[SO_O - 1 - o]
theo_nguong = set((el2n >= nguong).nonzero().squeeze(1).tolist())
assert toan_cuc <= theo_nguong
print(f"ngưỡng từ histogram: mỗi worker gửi {SO_O} số đếm, không gửi chỉ số nào;"
      f" giữ {len(theo_nguong):,} mẫu cho mục tiêu {K_GIU:,}")
do_tren_may("độ trùng Jaccard giữa chọn theo ngưỡng và chọn toàn cục",
            len(toan_cuc & theo_nguong) / len(toan_cuc | theo_nguong))
for ten, tap in (("toàn cục", toan_cuc), ("cục bộ", cuc_bo)):
    dem_lop = torch.bincount(y[torch.tensor(sorted(tap))], minlength=10).tolist()
    print(f"{ten:>9}: {len(tap):,} mẫu, theo lớp {dem_lop}")

Chọn cục bộ theo hạn ngạch cho một tập con khác hẳn: nó rẻ và không cần điều phối, nhưng giữ hạn ngạch
của từng shard thay cho thứ hạng toàn cục. Ở đây điều đó tình cờ trải đều các lớp hơn, vì mỗi shard gần
như là một lớp. Ở một cách chia khác, chính sự khác biệt ấy có thể bỏ sót có hệ thống một nhóm hiếm chỉ
thấy được trên toàn kho.

Cách thứ ba là cách chương nói cho lấy mẫu theo độ bất định phân tán: lấy ngưỡng toàn cục từ thống kê phân
phối điểm thay vì thứ hạng chính xác. Mỗi worker gửi 256 số đếm, và tập giữ lại gần như trùng với chọn
toàn cục, chỉ dư vài mẫu ở ô chứa ngưỡng. Chương vì vậy đặt hai phép thử cho mọi thiết kế phân tán: so chất
lượng tập chọn với một mốc tập trung, và so phụ trội chọn lọc với thời gian huấn luyện nó tiết kiệm.

## 6. Loại bỏ trùng lặp ở ba mức

Loại bỏ trùng lặp nhắm vào những mẫu thêm tính toán mà không thêm tín hiệu học, và không cần huấn luyện mô
hình nào. Cấy vào 10,000 ảnh hai loại bản sao: 300 bản **trùng từng byte**, và 300 bản **gần trùng**. Mỗi
bản gần trùng là một ảnh gốc bị thu nhỏ còn 20×20 rồi phóng lại 28×28, thêm nhiễu nhẹ, hoặc tối đi 15 phần
trăm: những thay đổi nhỏ kiểu đổi kích thước hay nén.

In [ ]:
g = torch.Generator().manual_seed(SEED)
SO_CAY = 300
goc_trung = torch.randperm(N, generator=g)[:SO_CAY]
goc_gan = torch.randperm(N, generator=g)[:SO_CAY]
ban_gan = anh[goc_gan].float()
kieu = torch.arange(SO_CAY) % 3
nho = F.interpolate(ban_gan[kieu == 0][:, None], size=(20, 20), mode="bilinear", align_corners=False)
ban_gan[kieu == 0] = F.interpolate(nho, size=(28, 28), mode="bilinear", align_corners=False)[:, 0]  # đổi cỡ
ban_gan[kieu == 1] += torch.randint(-12, 13, ban_gan[kieu == 1].shape, generator=g)                 # nhiễu nhẹ
ban_gan[kieu == 2] *= 0.85                                                                           # tối đi
ban_gan = ban_gan.round().clamp(0, 255).to(torch.uint8)
kho = torch.cat([anh, anh[goc_trung], ban_gan])                                       # 10,600 ảnh
nguon_goc = torch.cat([torch.full((N,), -1), goc_trung, goc_gan])                     # -1: ảnh gốc
print(f"kho sau khi cấy: {len(kho):,} ảnh")

# Mức 1: băm mật mã. Hai ảnh trùng mã băm khi và chỉ khi trùng từng byte.
nhom = {}
for i, a in enumerate(kho.numpy()):
    nhom.setdefault(hashlib.sha256(a.tobytes()).hexdigest(), []).append(i)
bi_bo = sorted(i for ds in nhom.values() for i in ds[1:])
print(f"SHA-256: {len(nhom):,} mã băm khác nhau, bỏ {len(bi_bo):,} bản trùng")
assert set(range(N, N + SO_CAY)) <= set(bi_bo)  # mọi bản trùng từng byte đều bị bắt
print(f"bản gần trùng bị SHA-256 bắt: {len(set(bi_bo) & set(range(N + SO_CAY, len(kho))))} trên {SO_CAY}")

SHA-256 bắt trọn 300 bản trùng từng byte và không bắt bản gần trùng nào: đổi một byte là đổi cả mã băm. Cách
này rẻ, tăng tuyến tính theo cỡ tập và song song hoá dễ, nên chương khuyên bắt đầu từ đây.

Mức 2 là **băm cảm nhận**: một chữ ký bền trước thay đổi nhỏ. Bản kinh điển, pHash, lấy biến đổi cosin rời
rạc (DCT) của ảnh, giữ 8×8 hệ số tần số thấp nhất trừ hệ số một chiều, rồi ghi 1 cho hệ số lớn hơn trung vị.
Tần số thấp là hình dáng tổng thể, thứ mà đổi cỡ, nhiễu hay độ sáng gần như không chạm tới. Hai ảnh gần
trùng khi hai chữ ký 63 bit khác nhau ở ít bit.

In [ ]:
from scipy.fft import dctn


def phash(a):
    he_so = dctn(a.float().numpy(), axes=(1, 2), norm="ortho")[:, :8, :8].reshape(len(a), 64)[:, 1:]
    bit = he_so > np.median(he_so, axis=1, keepdims=True)
    return (bit.astype(np.uint64) << np.arange(63, dtype=np.uint64)).sum(1)


chu_ky = phash(kho)
so_bit_khac = np.empty((len(kho), len(kho)), dtype=np.uint8)
for i in range(0, len(kho), 1_000):  # theo khối, để không cần cả ma trận uint64
    so_bit_khac[i:i + 1_000] = np.bitwise_count(chu_ky[i:i + 1_000, None] ^ chu_ky[None, :])
np.fill_diagonal(so_bit_khac, 255)

chi_so_gan = np.arange(N + SO_CAY, len(kho))
tong_cap_goc = N * (N - 1) // 2
print(f"{'ngưỡng':>6} {'bắt được bản gần trùng':>24} {'cặp ảnh gốc bị gắn cờ':>22}")
for nguong in (0, 2, 4, 6, 8):
    bat = (so_bit_khac[chi_so_gan, nguon_goc[chi_so_gan].numpy()] <= nguong).sum()
    cap_goc = int((so_bit_khac[:N, :N] <= nguong).sum() // 2)
    print(f"{nguong:>6} {bat:>18} / {SO_CAY} {cap_goc:>14,} / {tong_cap_goc:,}")
for k_, ten in {0: "đổi cỡ", 1: "nhiễu nhẹ", 2: "tối đi"}.items():
    d = so_bit_khac[chi_so_gan[kieu == k_], nguon_goc[chi_so_gan[kieu == k_]].numpy()]
    print(f"{ten:>10}: số bit khác với ảnh gốc, trung vị {int(np.median(d))}, lớn nhất {int(d.max())}")

In [ ]:
i_, j_ = np.nonzero(np.triu(so_bit_khac[:N, :N] == 0))
cap = list(zip(i_.tolist(), j_.tolist()))
da_xem, cap_xem = set(), []
for a_, b_ in cap:  # mỗi ảnh chỉ xuất hiện một lần trên hình
    if a_ not in da_xem and b_ not in da_xem:
        cap_xem.append((a_, b_))
        da_xem |= {a_, b_}
print(f"{len(cap)} cặp ảnh gốc có chữ ký trùng hẳn; xem {min(6, len(cap_xem))} cặp không chung ảnh nào")
fig, axes = plt.subplots(2, 6, figsize=(11, 5))
for c, (a_, b_) in enumerate(cap_xem[:6]):
    for hang, k_ in enumerate((a_, b_)):
        ax = axes[hang][c]
        ax.imshow(kho[k_], cmap="gray_r")
        ax.set_title(f"ảnh {k_:,}\n{TEN_LOP[y[k_]]}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
fig.suptitle("Cặp ảnh GỐC mà pHash coi là trùng (0 bit khác)", fontsize=11)
plt.tight_layout(h_pad=2.5)
plt.show()

Ngưỡng càng nới thì bắt được càng nhiều bản gần trùng đã cấy, nhưng cũng gắn cờ càng nhiều cặp ảnh gốc. Đổi
cỡ là thay đổi khó nhất trong ba loại, nên ngưỡng nào bắt hết nó cũng gắn cờ hàng chục nghìn cặp ảnh gốc. Hãy
nhìn các cặp ở trên: có cặp có lẽ là cùng một món đồ chụp lại, có cặp chỉ là hai món đồ cùng kiểu dáng. Chính
vì vậy chương dặn **ngưỡng gần trùng phải được kiểm định**, để không bỏ mất tín hiệu phân phối hữu ích. Mức
thứ ba là độ tương đồng embedding, như CLIP; nó tìm được ứng viên gần nghĩa, nhưng gần không có nghĩa là
trùng, và nó tốn tính toán hơn.

### MinHash cho văn bản gần trùng

Với văn bản, chương dùng MinHash kết hợp băm nhạy cục bộ (LSH) để xấp xỉ độ tương đồng Jaccard giữa hai tập
cụm từ. Mô phỏng một kho 2,000 tài liệu, mỗi tài liệu 60 từ, và cấy 100 bản sửa nhẹ: thay 3 trong 60 từ.

In [ ]:
rng = np.random.default_rng(SEED)
TU_VUNG = 3_000
SO_TL, DAI, SO_SUA = 2_000, 60, 100
tan_suat = 1 / np.arange(1, TU_VUNG + 1)  # vài từ rất phổ biến, rất nhiều từ hiếm, như văn bản thật
tai_lieu = [rng.choice(TU_VUNG, DAI, p=tan_suat / tan_suat.sum()) for _ in range(SO_TL)]
goc_sua = rng.choice(SO_TL, SO_SUA, replace=False)
for i in goc_sua:
    ban = tai_lieu[i].copy()
    ban[rng.choice(DAI, 3, replace=False)] = rng.choice(TU_VUNG, 3)
    tai_lieu.append(ban)
cap_that = {(int(i), SO_TL + k) for k, i in enumerate(goc_sua)}


def cum_tu(ds, n=3):
    """Tập các cụm 3 từ liên tiếp (shingle), băm về số nguyên 32 bit bằng crc32."""
    return {zlib.crc32(" ".join(map(str, ds[i:i + n])).encode()) for i in range(len(ds) - n + 1)}


def jaccard(a, b):
    return len(a & b) / len(a | b)


SO_HAM = 128
P = (1 << 61) - 1
he_so_a = rng.integers(1, P, SO_HAM, dtype=np.uint64)
he_so_b = rng.integers(0, P, SO_HAM, dtype=np.uint64)
cum = [np.fromiter(cum_tu(ds), dtype=np.uint64) for ds in tai_lieu]
# Mỗi hàm băm là (a·x + b) mod P; chữ ký là giá trị nhỏ nhất trên mọi cụm từ của tài liệu.
# Tính bằng số thực 128 bit là thừa; ở đây a·x tràn uint64 một cách tất định, đủ cho một hàm trộn.
chu_ky_vb = np.stack([((he_so_a[:, None] * c[None, :] + he_so_b[:, None]) % P).min(1) for c in cum])
print(f"chữ ký MinHash: {chu_ky_vb.shape[0]:,} tài liệu × {SO_HAM} số")

that = np.array([jaccard(cum_tu(tai_lieu[a]), cum_tu(tai_lieu[b])) for a, b in sorted(cap_that)])
uoc = np.array([(chu_ky_vb[a] == chu_ky_vb[b]).mean() for a, b in sorted(cap_that)])
print(f"{SO_SUA} cặp đã cấy: Jaccard thật trung bình {that.mean():.3f}, MinHash ước lượng {uoc.mean():.3f},"
      f" sai lệch tuyệt đối trung bình {np.abs(uoc - that).mean():.3f}")

# LSH: chia chữ ký thành 32 dải, mỗi dải 4 số; hai tài liệu là ứng viên nếu trùng trọn một dải.
DAI_LSH, HANG = 32, 4
ung_vien = set()
for d_ in range(DAI_LSH):
    xo = {}
    for i, khoa in enumerate(r.tobytes() for r in chu_ky_vb[:, d_ * HANG:(d_ + 1) * HANG]):
        xo.setdefault(khoa, []).append(i)
    for ds in xo.values():
        ung_vien |= {(a, b) for k, a in enumerate(ds) for b in ds[k + 1:]}
tong_cap = len(tai_lieu) * (len(tai_lieu) - 1) // 2
NGUONG_J = 0.5
tim_thay = {(a, b) for a, b in ung_vien if (chu_ky_vb[a] == chu_ky_vb[b]).mean() >= NGUONG_J}
print(f"so sánh mọi cặp: {tong_cap:,} cặp · LSH chỉ so {len(ung_vien):,} cặp ứng viên ({len(ung_vien) / tong_cap:.3%})")
print(f"cặp gần trùng tìm thấy: {len(tim_thay)}, trong đó đúng cặp đã cấy: {len(tim_thay & cap_that)} / {SO_SUA}")

Một bản sửa 3 trong 60 từ phá tối đa 9 trong 58 cụm 3 từ, nên Jaccard thật của nó vào khoảng ba phần tư, và
MinHash ước lượng con số đó chỉ bằng cách đếm số vị trí trùng trong hai chữ ký. LSH bỏ qua gần hết các cặp không có
dải nào trùng, nên số phép so sánh không còn tăng theo bình phương cỡ kho. Đó là lý do chương nói MinHash
với LSH "xấp xỉ độ tương đồng Jaccard một cách hiệu quả", và cũng là thứ cho phép mỗi worker tự tính chữ ký
trên shard của mình trong MinHash phân tán. Nó bắt được nội dung sửa nhẹ còn trùng nhiều, nhưng không chứng
minh được hai câu diễn đạt lại là tương đương về nghĩa.

## Thử thêm

1. Ở phần 2, thêm chính sách "bỏ 5 phần trăm EL2N cao nhất rồi giữ khó nhất": `lambda k: thu_tu[:-500][-k:]`.
   Nó đứng ở đâu giữa "khó nhất" và "ngẫu nhiên", và vì sao bỏ phần đỉnh lại giúp được?
2. Ở phần 3, chạy k-Center trên biểu diễn của chính mô hình thay thế, `torch.relu(proxy[0](X))`, thay cho PCA.
   Biểu diễn đã học từ nhãn có đổi thứ hạng giữa k-Center và chọn ngẫu nhiên không?
3. Ở phần 6, đổi `HANG` thành 8 (16 dải). Số cặp ứng viên và số cặp đã cấy bị bỏ sót thay đổi ra sao?

## Tóm lại

* Một mô hình thay thế nhỏ chấm được mọi mẫu trong vài epoch; EL2N và mất mát gần như cùng thứ hạng, còn số
  lần quên đo một thứ khác.
* Giữ "khó nhất" thuần tuý có thể dồn ngân sách vào vài lớp và vào nhãn sai; một rào chắn độ phủ theo lớp và
  một mốc ngẫu nhiên là hai thứ phải có trước khi tin một điểm số.
* k-Center thu nhỏ khoảng cách xa nhất bằng cách lấy điểm rìa, và không biết tới nhãn, nên độ phủ hình học
  chưa phải độ phủ của tác vụ.
* Bất đẳng thức chọn lọc phải đo bằng thời gian thật: chọn đắt cỡ một lần huấn luyện thì không bao giờ có lời,
  và lớp hiếm cần hạn ngạch tối thiểu vì chọn ngẫu nhiên không cứu được chúng.
* Băm mật mã bắt bản trùng từng byte với giá rẻ nhất, băm cảm nhận và MinHash bắt bản gần trùng nhưng cần một
  ngưỡng đã kiểm định.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Chọn lọc dữ liệu — bớt việc trước khi việc bắt đầu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch09-data-selection/), dựng từ chương
[*Data Selection*](https://mlsysbook.ai/vol1/data_selection/data_selection.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.